<a href="https://colab.research.google.com/github/Sanidhya0507/laliga-match-predictor/blob/main/laliga_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# La Liga Match Outcome Predictor


In [2]:
import pandas as pd

# Season codes: "1920" = 2019-20, ..., "2627" = current season
seasons = ["1920", "2021", "2122", "2223", "2324", "2425", "2526", "2627"]

frames = []
for s in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{s}/SP1.csv"
    df_season = pd.read_csv(url)
    df_season["Season"] = s
    frames.append(df_season)

df = pd.concat(frames, ignore_index=True)

# Keep only the columns we need for now
cols = ["Season", "Date", "HomeTeam", "AwayTeam",
        "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]
df = df[cols]

print(df.shape)
df.tail()



(2749, 10)


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A
2744,2627,18/10/2026,Celta,Alaves,2,1,H,2.10,3.4,3.50
2745,2627,18/10/2026,Malaga,Sociedad,2,5,A,3.30,3.5,2.15
2746,2627,18/10/2026,Real Madrid,Sevilla,4,1,H,1.25,7.0,9.50
2747,2627,19/10/2026,Getafe,Vallecano,1,1,D,2.90,3.0,2.70
2748,2627,21/10/2026,Levante,Ath Bilbao,3,1,H,3.40,3.5,2.15


In [3]:
# 1. Turn date text into real dates (day comes first)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

# 2. Sort oldest -> newest
df = df.sort_values("Date").reset_index(drop=True)

# 3. Any missing values?
print(df.isna().sum())

# 4. Exact team name spellings
print(sorted(df["HomeTeam"].unique()))

# 5. Every Clasico in our data
clasico = df[
    ((df["HomeTeam"] == "Barcelona") & (df["AwayTeam"] == "Real Madrid")) |
    ((df["HomeTeam"] == "Real Madrid") & (df["AwayTeam"] == "Barcelona"))
]
clasico

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
B365H       0
B365D       0
B365A       0
dtype: int64
['Alaves', 'Almeria', 'Ath Bilbao', 'Ath Madrid', 'Barcelona', 'Betis', 'Cadiz', 'Celta', 'Eibar', 'Elche', 'Espanol', 'Getafe', 'Girona', 'Granada', 'Huesca', 'La Coruna', 'Las Palmas', 'Leganes', 'Levante', 'Malaga', 'Mallorca', 'Osasuna', 'Oviedo', 'Real Madrid', 'Santander', 'Sevilla', 'Sociedad', 'Valencia', 'Valladolid', 'Vallecano', 'Villarreal']


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A
169,1920,2019-12-18,Barcelona,Real Madrid,0,0,D,1.72,4.2,4.33
254,1920,2020-03-01,Real Madrid,Barcelona,2,0,H,2.20,3.6,3.10
437,2021,2020-10-24,Barcelona,Real Madrid,1,3,A,2.00,3.6,3.60
671,2021,2021-04-10,Real Madrid,Barcelona,2,1,H,2.80,3.8,2.30
854,2122,2021-10-24,Barcelona,Real Madrid,1,2,A,2.75,3.6,2.45
1044,2122,2022-03-20,Real Madrid,Barcelona,0,4,A,2.15,3.6,3.25
1226,2223,2022-10-16,Real Madrid,Barcelona,3,1,H,2.20,3.6,3.10
1399,2223,2023-03-19,Barcelona,Real Madrid,2,1,H,2.15,3.5,3.25
1621,2324,2023-10-28,Barcelona,Real Madrid,1,2,A,2.55,3.6,2.60
1838,2324,2024-04-21,Real Madrid,Barcelona,3,2,H,1.85,3.8,4.00


In [4]:
K = 20          # how fast ratings react
HOME_ADV = 60   # home advantage in Elo points

elo = {}        # team name -> current rating
home_elo_before = []
away_elo_before = []

for i, row in df.iterrows():
    h = row["HomeTeam"]
    a = row["AwayTeam"]
    r_h = elo.get(h, 1500)   # new teams start at 1500
    r_a = elo.get(a, 1500)

    # Save ratings BEFORE the match (these become our features)
    home_elo_before.append(r_h)
    away_elo_before.append(r_a)

    # Expected score for the home team
    exp_h = 1 / (1 + 10 ** ((r_a - (r_h + HOME_ADV)) / 400))

    # Actual score for the home team
    if row["FTR"] == "H":
        s_h = 1
    elif row["FTR"] == "D":
        s_h = 0.5
    else:
        s_h = 0

    # Update both teams (what one gains, the other loses)
    change = K * (s_h - exp_h)
    elo[h] = r_h + change
    elo[a] = r_a - change

df["HomeElo"] = home_elo_before
df["AwayElo"] = away_elo_before
df["EloDiff"] = df["HomeElo"] + HOME_ADV - df["AwayElo"]

# Current top 10
print(pd.Series(elo).sort_values(ascending=False).head(10))

Barcelona      1816.968915
Real Madrid    1752.726309
Ath Madrid     1673.265459
Betis          1628.967325
Villarreal     1600.819242
Vallecano      1536.494752
Sociedad       1535.889040
Celta          1535.027746
Ath Bilbao     1518.619099
La Coruna      1501.857892
dtype: float64


In [5]:
from collections import defaultdict

history = defaultdict(list)   # team -> list of points from past matches
home_form = []
away_form = []

for i, row in df.iterrows():
    h = row["HomeTeam"]
    a = row["AwayTeam"]

    # Form BEFORE this match (last 5 games)
    recent_h = history[h][-5:]
    recent_a = history[a][-5:]
    home_form.append(sum(recent_h) / len(recent_h) if recent_h else 1.0)
    away_form.append(sum(recent_a) / len(recent_a) if recent_a else 1.0)

    # Points earned in this match
    if row["FTR"] == "H":
        ph, pa = 3, 0
    elif row["FTR"] == "D":
        ph, pa = 1, 1
    else:
        ph, pa = 0, 3

    history[h].append(ph)
    history[a].append(pa)

df["HomeForm"] = home_form
df["AwayForm"] = away_form
df["FormDiff"] = df["HomeForm"] - df["AwayForm"]

# Check: each team's form going into its latest match
df[(df["HomeTeam"].isin(["Barcelona", "Real Madrid"])) |
   (df["AwayTeam"].isin(["Barcelona", "Real Madrid"]))].tail(6)[
    ["Date", "HomeTeam", "AwayTeam", "FTR", "HomeForm", "AwayForm"]]

,Date,HomeTeam,AwayTeam,FTR,HomeForm,AwayForm
2723,2026-09-19,Sevilla,Barcelona,A,2.0,3.0
2728,2026-09-20,Ath Madrid,Real Madrid,H,2.0,2.4
2731,2026-10-10,Barcelona,Getafe,H,3.0,1.0
2732,2026-10-10,Real Madrid,Villarreal,H,1.8,1.2
2739,2026-10-17,Betis,Barcelona,A,2.6,3.0
2746,2026-10-18,Real Madrid,Sevilla,H,1.8,1.6


In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss

features = ["EloDiff", "HomeForm", "AwayForm"]

train = df[df["Season"].isin(["2021", "2122", "2223", "2324", "2425"])]
test = df[df["Season"].isin(["2526", "2627"])]
test = test.dropna(subset=["B365H", "B365D", "B365A"])

X_train, y_train = train[features], train["FTR"]
X_test, y_test = test[features], test["FTR"]

# Scale features (EloDiff is in hundreds, form is 0-3), then fit
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)
print("Classes order:", model.classes_)
print("Model accuracy:", round(accuracy_score(y_test, pred), 3))
print("Model log loss:", round(log_loss(y_test, proba), 3))

# Baseline 1: always home
print("Always-home accuracy:", round((y_test == "H").mean(), 3))

# Baseline 2: bookmaker odds -> probabilities (order A, D, H)
implied = 1 / test[["B365A", "B365D", "B365H"]]
implied = implied.div(implied.sum(axis=1), axis=0)
book_pred = np.array(["A", "D", "H"])[implied.values.argmax(axis=1)]
print("Bookmaker accuracy:", round(accuracy_score(y_test, book_pred), 3))
print("Bookmaker log loss:", round(log_loss(y_test, implied.values, labels=["A", "D", "H"]), 3))

# How often does the model predict each outcome?
print(pd.Series(pred).value_counts())

Classes order: ['A' 'D' 'H']
Model accuracy: 0.514
Model log loss: 0.975
Always-home accuracy: 0.48
Bookmaker accuracy: 0.546
Bookmaker log loss: 0.957
H    334
A    132
D      3
Name: count, dtype: int64


In [7]:
# Retrain on everything except the warm-up season
full = df[df["Season"] != "1920"]
final_model = make_pipeline(StandardScaler(), LogisticRegression())
final_model.fit(full[features], full["FTR"])

# Build the Clasico row from the CURRENT ratings and form
h, a = "Barcelona", "Real Madrid"
recent_h = history[h][-5:]
recent_a = history[a][-5:]

clasico_row = pd.DataFrame([{
    "EloDiff": elo[h] + HOME_ADV - elo[a],
    "HomeForm": sum(recent_h) / len(recent_h),
    "AwayForm": sum(recent_a) / len(recent_a),
}])
print(clasico_row)

p = final_model.predict_proba(clasico_row[features])[0]
labels = {"H": "Barcelona win", "D": "Draw", "A": "Real Madrid win"}
for cls, prob in zip(final_model.classes_, p):
    print(f"{labels[cls]}: {prob:.1%}")

      EloDiff  HomeForm  AwayForm
0  124.242606       3.0       2.4
Real Madrid win: 25.9%
Draw: 18.7%
Barcelona win: 55.4%


## El Clásico Prediction (locked in before kickoff)

**Made:** Saturday, October 24, 2026, 7:24 pm CT
**Match:** Barcelona vs Real Madrid, Camp Nou, Sunday Oct 25, 2026, 3:00 pm CT

| Outcome | Probability |
|---|---|
| Barcelona win | 55.4% |
| Draw | 18.7% |
| Real Madrid win | 25.9% |

**Inputs:** EloDiff = 124.2 · Barça form = 3.0 · Madrid form = 2.4

**Result:** _to be added after the match_